In [1]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
from vnpy.alpha.lab import AlphaLab
from vnpy.trader.constant import Interval
from vnpy.alpha import Segment, AlphaDataset
from vnpy.alpha.dataset.template import logger
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path
from datetime import datetime
import gc

In [2]:
# ============================================================================
# Cell 2: 配置
# ============================================================================
vt_index_symbol = "000300.SSE"
BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'

# 获取 MF_Lab
lab = AlphaLab(str(LAB_PATH))

# 时间配置
start = datetime(2018, 1, 1)
end = datetime(2026, 5, 8)

# 训练 / 验证 / 测试
DATASET_NAME = 'v100'
n_quantiles = 30

In [3]:
# ============================================================================
# Cell 3: 加载数据集
# ============================================================================
dataset: AlphaDataset = lab.load_dataset(DATASET_NAME)

In [4]:
# ============================================================================
# Cell 4: 提取 LambdaRank 数据
# ============================================================================
logger.info('提取 LambdaRank 训练数据...')
X_train, y_train, meta_train, group_train = dataset.extract_lambdarank_data(
    Segment.TRAIN, n_quantiles=n_quantiles
)

logger.info('提取验证数据...')
X_valid, y_valid, meta_valid, group_valid = dataset.extract_lambdarank_data(
    Segment.VALID, n_quantiles=n_quantiles
)

logger.info('提取测试数据...')
X_test, y_test, meta_test, group_test = dataset.extract_lambdarank_data(
    Segment.TEST, n_quantiles=n_quantiles
)

2026-05-27 15:55:41 提取 LambdaRank 训练数据...
2026-05-27 15:55:42 TRAIN, 标签值: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
2026-05-27 15:55:42 TRAIN, X.shape=(437100, 65)
2026-05-27 15:55:42 提取验证数据...
2026-05-27 15:55:42 VALID, 标签值: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
2026-05-27 15:55:42 VALID, X.shape=(72600, 65)
2026-05-27 15:55:42 提取测试数据...
2026-05-27 15:55:42 TEST, 标签值: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
2026-05-27 15:55:42 TEST, X.shape=(95100, 65)


In [9]:
# ============================================================================
# Cell 5: 训练 LambdaRank 模型
# ============================================================================
logger.info('开始训练 LambdaRank 模型...')

# X_* 为 pd.DataFrame
train_data = lgb.Dataset(X_train, label=y_train, group=group_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, group=group_valid, reference=train_data)

params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [5, 10],   # 只要有一个指标在stopping_round轮内没有提升就会停
    'label_gain': [i**2 for i in range(n_quantiles)],
    'lambdarank_truncation_level': 100,

    'num_leaves': 1024,
    'max_depth': -1,
    'min_data_in_leaf': 300,

    'learning_rate': 0.001,
    'feature_fraction': 0.88,
    'bagging_fraction': 0.87,
    'bagging_freq': 5,

    'lambda_l1': 30,
    'lambda_l2': 0.0,

    'boosting_type': 'gbdt',
    'device': 'gpu',
    'verbose': -1,
    'seed': 42,
    'num_threads': -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(100, first_metric_only=True),
        lgb.log_evaluation(period=1),
    ]
)

logger.info(f'训练完成！最佳迭代轮数: {model.best_iteration}')
if model.best_score:
    logger.info(f'VALID BEST NDCG: {model.best_score['valid']}')
    logger.info(f'TRAIN NDCG: {model.best_score['train']}')

2026-05-27 16:49:50 开始训练 LambdaRank 模型...
[1]	train's ndcg@5: 0.355839	train's ndcg@10: 0.361915	valid's ndcg@5: 0.307482	valid's ndcg@10: 0.319707
Training until validation scores don't improve for 100 rounds
[2]	train's ndcg@5: 0.380379	train's ndcg@10: 0.378468	valid's ndcg@5: 0.31175	valid's ndcg@10: 0.328056
[3]	train's ndcg@5: 0.397611	train's ndcg@10: 0.395253	valid's ndcg@5: 0.331086	valid's ndcg@10: 0.341249
[4]	train's ndcg@5: 0.403023	train's ndcg@10: 0.400907	valid's ndcg@5: 0.339067	valid's ndcg@10: 0.339068
[5]	train's ndcg@5: 0.404088	train's ndcg@10: 0.401336	valid's ndcg@5: 0.344973	valid's ndcg@10: 0.343436
[6]	train's ndcg@5: 0.406715	train's ndcg@10: 0.404244	valid's ndcg@5: 0.349849	valid's ndcg@10: 0.345841
[7]	train's ndcg@5: 0.418849	train's ndcg@10: 0.413484	valid's ndcg@5: 0.365112	valid's ndcg@10: 0.355077
[8]	train's ndcg@5: 0.418057	train's ndcg@10: 0.414526	valid's ndcg@5: 0.367934	valid's ndcg@10: 0.355156
[9]	train's ndcg@5: 0.418966	train's ndcg@10: 0.4

In [10]:
# ============================================================================
# Cell 6: 生成回测信号
# ============================================================================
logger.info('---在测试集上预测---')

predictions = model.predict(X_test, num_iteration=model.best_iteration)
logger.info(f'预测完成，预测样本数:{len(predictions)}')

signal = meta_test.with_columns([
    pl.Series('signal', predictions)
])


logger.info(f'siganl.shape: {signal.shape}')
logger.info('siganl:')
logger.info(signal.head(5))
logger.info(signal.tail(5))

2026-05-27 16:50:32 ---在测试集上预测---
2026-05-27 16:50:32 预测完成，预测样本数:95100
2026-05-27 16:50:32 siganl.shape: (95100, 3)
2026-05-27 16:50:32 siganl:
2026-05-27 16:50:32 shape: (5, 3)
┌─────────────────────┬─────────────┬───────────┐
│ datetime            ┆ vt_symbol   ┆ signal    │
│ ---                 ┆ ---         ┆ ---       │
│ datetime[μs]        ┆ str         ┆ f64       │
╞═════════════════════╪═════════════╪═══════════╡
│ 2025-01-02 00:00:00 ┆ 000001.SZSE ┆ -0.001831 │
│ 2025-01-02 00:00:00 ┆ 000002.SZSE ┆ -0.000607 │
│ 2025-01-02 00:00:00 ┆ 000063.SZSE ┆ 0.0       │
│ 2025-01-02 00:00:00 ┆ 000100.SZSE ┆ -0.002742 │
│ 2025-01-02 00:00:00 ┆ 000157.SZSE ┆ -0.004354 │
└─────────────────────┴─────────────┴───────────┘
2026-05-27 16:50:32 shape: (5, 3)
┌─────────────────────┬────────────┬───────────┐
│ datetime            ┆ vt_symbol  ┆ signal    │
│ ---                 ┆ ---        ┆ ---       │
│ datetime[μs]        ┆ str        ┆ f64       │
╞═════════════════════╪════════════╪══════

In [11]:
predictions_IS = model.predict(X_train, num_iteration=model.best_iteration)
signal_IS = meta_train.with_columns([
    pl.Series('signal', predictions_IS)
])

predictions_VA = model.predict(X_valid, num_iteration=model.best_iteration)
signal_VA = meta_valid.with_columns([
    pl.Series('signal', predictions_VA)
])

X_TV = pd.concat([X_train, X_valid])
meta_TV = pl.concat([meta_train, meta_valid])
predictions_TV = model.predict(X_TV, num_iteration=model.best_iteration)
signal_TV= meta_TV.with_columns([
    pl.Series('signal', predictions_TV)
])

X_ALL = pd.concat([X_train, X_valid, X_test])
meta_ALL = pl.concat([meta_train, meta_valid, meta_test])
predictions_ALL = model.predict(X_ALL, num_iteration=model.best_iteration)
signal_ALL = meta_ALL.with_columns([
    pl.Series('signal', predictions_ALL)
])

In [12]:
# ============================================================================
# Cell 7: 保存模型和信号
# ============================================================================
MODEL_NAME = 'v100'
SIGNAL_NAME = 'v100'

lab.save_model(MODEL_NAME, model)
lab.save_signal(SIGNAL_NAME, signal, 0)
lab.save_signal(SIGNAL_NAME, signal_IS, 1)
lab.save_signal(SIGNAL_NAME, signal_VA, 2)
lab.save_signal(SIGNAL_NAME, signal_TV, 3)
lab.save_signal(SIGNAL_NAME, signal_ALL, 4)
logger.info('模型和信号已保存')

2026-05-27 16:50:54 模型和信号已保存


In [9]:
# ============================================================================
# Cell 8: 特征重要性
# ============================================================================
logger.info('---特征重要性---')

importance_df = pd.DataFrame({
    'feature': model.feature_name(),
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

logger.info('Top 20 :')
print(importance_df.head(20))

2026-05-26 00:03:46 ---特征重要性---
2026-05-26 00:03:46 Top 20 :
                  feature   importance
60          ma_divergence  1686.296751
29                   rstr  1324.524717
45  range_adjusted_amihud  1287.528789
38           abn_turnover   803.444044
58              macd_hist   608.471042
59             boll_width   603.675684
31           streverse_2m   368.663969
43             liq_amihud   272.407820
42     abn_turnover_accel   239.912011
62            daily_range   184.641382
51                   rmax   155.882841
47          vol_stability   142.638554
40                    gtr   120.866951
39           turnover_vol   108.861396
34               rvol_21d   106.797778
30           streverse_1m    94.361963
63       effective_spread    84.731488
41  turnover_ret_interact    82.707159
56           shadow_ratio    81.002447
49                  rskew    80.646499


In [10]:
# ============================================================================
# Cell 9: 查看完整重要性
# ============================================================================
with pd.option_context('display.max_rows', None):
    print(importance_df)

                      feature   importance
60              ma_divergence  1686.296751
29                       rstr  1324.524717
45      range_adjusted_amihud  1287.528789
38               abn_turnover   803.444044
58                  macd_hist   608.471042
59                 boll_width   603.675684
31               streverse_2m   368.663969
43                 liq_amihud   272.407820
42         abn_turnover_accel   239.912011
62                daily_range   184.641382
51                       rmax   155.882841
47              vol_stability   142.638554
40                        gtr   120.866951
39               turnover_vol   108.861396
34                   rvol_21d   106.797778
30               streverse_1m    94.361963
63           effective_spread    84.731488
41      turnover_ret_interact    82.707159
56               shadow_ratio    81.002447
49                      rskew    80.646499
61             boll_bandwidth    75.711359
37                       cmra    65.042337
36         